In [1]:
import sys, importlib
MODULE_DIR = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script"
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import tommo_ld_tools
importlib.reload(tommo_ld_tools)

<module 'tommo_ld_tools' from '/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script/tommo_ld_tools.py'>

In [2]:
gwas_path = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/analysis/assoc_plink2/results/02.summary_vis/gwas_summary.plink2.csv"
tommo_ld_dir = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/ToMMo_60KJPN/co-occurrence"

In [3]:
# 提取TEST列包含'ADD'的ID形成list
import ast
import pandas as pd

gwas_df = pd.read_csv(gwas_path)

# 解析字符串为列表后检查
def contains_add(test_str):
    try:
        test_list = ast.literal_eval(test_str)
        return 'ADD' in test_list
    except:
        return False

variant_ids_method = gwas_df[gwas_df['TEST'].apply(contains_add)]['ID'].tolist()
variant_ids = variant_ids_method

print(f"包含'ADD'的变异ID数量: {len(variant_ids)}")
print(f"变异ID列表: {variant_ids}")

包含'ADD'的变异ID数量: 5
变异ID列表: ['chr3:154069965:A:G', 'chr16:53884344:G:A', 'chr16:53887925:T:C', 'chr17:13528059:G:A', 'chr17:13528180:G:A']


In [4]:
# 将variant_ids写入TSV文件
import os

# 创建TSV文件路径（保存到当前工作目录）
tsv_path = "variant_ids_with_ADD.tsv"

# 将variant_ids写入TSV文件
variant_df = pd.DataFrame(variant_ids)
variant_df.to_csv(tsv_path, sep='\t', index=False, header=False)

print(f"变异ID已写入文件: {tsv_path}")
print(f"文件包含 {len(variant_ids)} 个变异ID")

# 返回文件路径
focus_loci_path = tsv_path

变异ID已写入文件: variant_ids_with_ADD.tsv
文件包含 5 个变异ID


In [5]:
from tommo_ld_tools import extract_ld_from_tommo_for_focus_loci 

tommo_ld_dict_path, tommo_ld_log = extract_ld_from_tommo_for_focus_loci(focus_loci_path, tommo_ld_dir)

tommo_ld_log

,ID,CHR,POS,Match_Found,Num_Matches
0,chr3:154069965:A:G,chr3,154069965,True,24
1,chr16:53884344:G:A,chr16,53884344,True,95
2,chr16:53887925:T:C,chr16,53887925,True,95
3,chr17:13528059:G:A,chr17,13528059,True,50
4,chr17:13528180:G:A,chr17,13528180,True,51


In [6]:
wgs_bed_prefix = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/wgs/19.tommo_panel_filter/cteph_agp3k.lowfreq_common"
case_prefix = "PHOM"
plink2_path = "/home/b/b37974/plink2_alpha6/plink2"
tommo_ld_dict = pd.read_pickle(tommo_ld_dict_path)

In [7]:
from tommo_ld_tools import compute_ld_between_focus_and_tommo_linked_variants

focus_ld_dict_path = compute_ld_between_focus_and_tommo_linked_variants(
    tommo_dict=tommo_ld_dict,
    bed_prefix=wgs_bed_prefix,
    case_prefix=case_prefix,
    plink2_path=plink2_path,
    threads=6
)

PLINK v2.0.0-a.6.20LM 64-bit Intel (7 Jul 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/02.tommo_ld_compare/tmp/case_data.log.
Options in effect:
  --bfile /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/wgs/19.tommo_panel_filter/cteph_agp3k.lowfreq_common
  --keep /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/02.tommo_ld_compare/tmp/case.keep
  --make-bed
  --out /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/02.tommo_ld_compare/tmp/case_data
  --threads 6

Start time: Wed Oct  8 09:59:42 2025
515039 MiB RAM detected, ~401912 available; reserving 257519 MiB for main
workspace.
Using up to 6 compute threads.
3018 samples (1934 females, 1084 males; 3018 founders) loaded from
/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/wgs/19.tommo_panel_filter/cteph_agp3k.l

In [8]:
from tommo_ld_tools import merge_tommo_and_focus_ld_dict

merged_dict_path = merge_tommo_and_focus_ld_dict(tommo_ld_dict_path, focus_ld_dict_path, select='control') # type: ignore

In [9]:
from tommo_ld_tools import plot_ld_comparison_from_merged_dict

plot_ld_comparison_from_merged_dict(merged_dict_path, select='control') # type: ignore

'/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/02.tommo_ld_compare/tommo_vs_focus_ld_scatter.pdf'